# 1 &mdash; Lorenz 63: an autonomous chaotic system

Demonstration of **Section 4.1** of [*B-LSTM-MIONet: Bayesian LSTM-based Neural Operators for
Learning the Response of Complex Dynamical Systems to Length-Variant Multiple Input
Functions*](https://arxiv.org/abs/2311.16519) (Kong, Mollaali, Moya, Lu and Lin, arXiv:2311.16519).
The Lorenz 63 system is **autonomous**, so there is no control function $u$: following Eq. (5) of
the paper, the LSTM branch is fed the *history of the state itself*, $x_{(t_0,t_n]}$, the state
branch sees the current state $x(t_n)$, and the trunk sees the local time spacing $h(t_n)$; the
operator predicts $x(t_n + h(t_n))$. The architecture is Figure 3 of the paper, reproduced in
`assets/paper/fig3_lstm_mionet_architecture.png`.

Every step below uses the `blstm_mionet` Python API, and the markdown cell before each step shows
the equivalent `blstm-mionet` command line call.

In [ ]:
import logging
import os
import warnings
from pathlib import Path

# Locate the repository root, whether the kernel started in notebooks/ or at the root.
ROOT = Path.cwd().resolve()
while not (ROOT / "configs" / "lorentz.yaml").is_file():
    if ROOT == ROOT.parent:
        raise RuntimeError("could not find the repository root (no configs/lorentz.yaml above cwd)")
    ROOT = ROOT.parent

# Paths inside a configuration file are relative to the working directory, so work from the
# repository root: data/, mlruns/ and figures/ are then created there (all three are git-ignored).
os.chdir(ROOT)
os.environ["MLFLOW_TRACKING_URI"] = str(ROOT / "mlruns")
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"  # set before mlflow is imported
warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)

print("repository root     :", ROOT)
print("MLFLOW_TRACKING_URI :", os.environ["MLFLOW_TRACKING_URI"])

In [ ]:
import copy

import matplotlib.pyplot as plt
import numpy as np
import torch

from blstm_mionet.config import load_config
from blstm_mionet.data.datasets import prepare_torch_dataset, split_dataset
from blstm_mionet.data.generate import generate_trajectories, load_dataset, save_dataset
from blstm_mionet.evaluation.evaluate import evaluate_recursive, evaluate_single_step
from blstm_mionet.models import build_model, dataset_preparer_for
from blstm_mionet.training import tracking
from blstm_mionet.training.trainer import train_adam
from blstm_mionet.utils.device import resolve_device
from blstm_mionet.utils.seed import set_seed

# blstm_mionet.evaluation.plotting selects the non-interactive "Agg" backend when it is
# imported, so ask for the inline backend again afterwards.
%matplotlib inline

print("torch               :", torch.__version__)

## Settings: `QUICK` versus the paper

`QUICK = True` shrinks every dimension of the experiment so that the whole notebook runs in about
two minutes on a laptop CPU. `QUICK = False` uses `configs/lorentz.yaml` exactly as shipped, which
*is* the paper's Section 4.1 setting; that needs a GPU and several hours, so prefer the command
lines collected in the last cell.

| | `QUICK` | paper, Sec. 4.1 (`configs/lorentz.yaml`) |
| --- | --- | --- |
| trajectories | 40 | 5000 initial values from $X_0 = [-17,20]\times[-23,28]\times[0,50]$ |
| horizon $T$ | 2 s | 20 s |
| RK-4 step $\Delta$ | 0.01 s | 0.01 s |
| masks per trajectory | 40 | 4, giving $N_{\text{train}} = 20000$ |
| max spacing $h_{\max}$ | 0.02 s (`search_len = 2`) | 0.02 s |
| branch / trunk | width 100, depth 3 | width 200, depth 3 |
| LSTM | 10 units, 1 layer | 10 units, 2 layers (the published model) |
| epochs | 5 | 1000, early stopping after 80 |
| test sub-sequences | 20 per trajectory | 200 per trajectory, 100 trajectories |

Everything a run produces goes to `data/`, `mlruns/` and `figures/` at the repository root.

In [ ]:
QUICK = True

QUICK_OVERRIDES = [
    # data: 40 trajectories of 2 s instead of 5000 of 20 s
    "data.n_sample=40",
    "data.t_max=2.0",
    "data.output=data/lorentz_demo_train.npy",
    # a smaller operator
    "model.branch_state.width=100",
    "model.branch_memory.width=100",
    "model.branch_memory.lstm_layer_num=1",
    "model.trunk.width=100",
    # training: 5 epochs on the CPU, and no entry in the MLflow model registry
    "training.datafile=data/lorentz_demo_train.npy",
    "training.search_num=40",
    "training.batch_size=64",
    "training.epochs=5",
    "training.device=cpu",
    "training.run_name=lorentz_quick",
    "training.registered_model_name=null",
    "training.plot_trajs=false",
    # inference: 20 sub-sequences per test trajectory instead of 200
    "inference.search_num=20",
    "inference.device=cpu",
    "inference.plot_trajs=false",
]

# Paper scale: configs/lorentz.yaml already carries the Section 4.1 values, so only the
# device has to be pointed at the GPU.
FULL_OVERRIDES = [
    "training.device=0",
    "inference.device=0",
]

config = load_config(
    ROOT / "configs" / "lorentz.yaml", QUICK_OVERRIDES if QUICK else FULL_OVERRIDES
)

TRAIN_FILE = config.data.output
TEST_FILE = "data/lorentz_demo_test.npy" if QUICK else "data/lorentz_N_100_h001_T20.npy"
ROLLOUT_FILE = "data/lorentz_demo_rollout.npy" if QUICK else "data/lorentz_N_1_h001_T20.npy"

print(f"architecture  : {config.model.architecture}")
print(f"trajectories  : {config.data.n_sample} x T = {config.data.t_max} s at dt = {config.data.step_size} s")
print(f"masks         : {config.training.search_num} per trajectory")
print(f"h_max         : {config.training.search_len * config.data.step_size} s")
print(f"epochs        : {config.training.epochs}")

## 1. Generate the trajectories

`generate_trajectories` integrates the Lorenz vector field with RK-4 at a fixed step
$\Delta = 0.01$ s and returns a dictionary with `x` of shape `[N_time, C, N_sample]` and `t` of
shape `[N_time, 1]`. Autonomous systems store no `u`. Three datasets are built here: one to train
on, one to test on, and a single trajectory used for the recursive rollout at the end.

```bash
blstm-mionet generate --config configs/lorentz.yaml \
    --n-sample 40 --set data.t_max=2.0 --output data/lorentz_demo_train.npy
```

In [ ]:
def make_dataset(base, n_sample, seed, output):
    # Integrate ``n_sample`` trajectories with ``seed`` and save them to ``output``.
    cfg = copy.deepcopy(base)
    cfg.n_sample, cfg.seed, cfg.output = n_sample, seed, output
    set_seed(cfg.seed)
    data = generate_trajectories(cfg)
    save_dataset(data, cfg.output)
    print(f"{output}: x {data['x'].shape}, t {data['t'].shape}")
    return data


train_raw = make_dataset(config.data, config.data.n_sample, 999, TRAIN_FILE)
test_raw = make_dataset(config.data, 10 if QUICK else 100, 2024, TEST_FILE)
rollout_raw = make_dataset(config.data, 1, 7, ROLLOUT_FILE)

## 2. A look at the data and at the masking (paper Sec. 3.4)

Section 3.4 of the paper turns one trajectory into many training quartets
$\{x(t_n),\; x_{(t_0,t_n]},\; h(t_n),\; x(t_n + h(t_n))\}$ by *masking*: the trajectory is
replicated `search_num` times and everything after a randomly drawn cut $t_n$ is set to zero,

$$x_{(t_0,t_1]} = [x(t_0), x(t_1), 0, \dots, 0], \quad \dots, \quad
  x_{(t_0,t_n]} = [x(t_0), \dots, x(t_n)].$$

The zeros are exactly what `LSTM_MLP` uses as the padding mask (`pack_padded_sequence` on the
count of non-zero entries), so one batch can hold histories of many different lengths. The spacing
$h(t_n)$ is drawn from $[0, h_{\max}]$ and the target $x(t_n+h(t_n))$ is interpolated with a cubic
spline between the grid points.

`prepare_local_predict_dataset` implements this and returns the four arrays below; it is selected
automatically from the architecture by `dataset_preparer_for`. There is no CLI equivalent: this
step runs inside `blstm-mionet train` and `blstm-mionet infer`.

In [ ]:
preparer = dataset_preparer_for(config.model.architecture)
print(f"{config.model.architecture} is prepared by {preparer.__name__}()\n")

train_split, _ = split_dataset(train_raw, test_size=0.0)
input_masked, x_n, x_next, t_params = preparer(
    data=train_split,
    state_component=config.training.state_component,
    search_len=config.training.search_len,
    search_num=config.training.search_num,
    search_random=config.training.search_random,
    offset=config.training.offset,
    t_max=config.training.t_max,
)

n_traj = train_split[1].shape[0]
t_grid = np.asarray(train_split[2]).squeeze()
dt = float(t_grid[1] - t_grid[0])
print(f"\n{input_masked.shape[0]} quartets = {config.training.search_num} masks x {n_traj} trajectories")
print(f"t_params columns are (t_n, h, t_n + h) in seconds, e.g. {np.round(t_params[0], 4)}")

In [ ]:
SAMPLE = 0
picks = [s * n_traj + SAMPLE for s in range(3)]
colours = ["tab:blue", "tab:orange", "tab:green"]

fig, ax = plt.subplots(figsize=(9.5, 4.2))
ax.plot(
    t_grid,
    train_split[1][SAMPLE, :, config.training.state_component],
    color="0.8", lw=1.0, label="full trajectory $x(t)$",
)
for colour, k in zip(colours, picks, strict=True):
    t_n, h, t_end = t_params[k]
    n_hist = int(round(t_n / dt)) + 1
    ax.plot(t_grid[:n_hist], input_masked[k, :n_hist, 0], color=colour, lw=1.6,
            label=f"masked history, $t_n$ = {t_n:.2f} s")
    ax.plot(t_n, x_n[k, 0], "o", color=colour, ms=7)
    ax.plot(t_end, x_next[k, 0], "*", color=colour, ms=13)
ax.set_xlabel("$t$ (s)")
ax.set_ylabel("$x(t)$")
ax.set_title("Three masks of the same Lorenz trajectory (paper Sec. 3.4)\n"
             "circle: current state $x(t_n)$   star: target $x(t_n+h)$")
ax.legend(fontsize=8, loc="best")
fig.tight_layout()
plt.show()

print(f"{'t_n (s)':>10} {'h (s)':>9} {'x(t_n)':>9} {'x(t_n+h)':>10} {'non-zero steps':>15}")
for k in picks:
    t_n, h, _ = t_params[k]
    print(f"{t_n:10.3f} {h:9.4f} {x_n[k, 0]:9.3f} {x_next[k, 0]:10.3f} "
          f"{int(np.count_nonzero(input_masked[k])):15d}")

## 3. Build the operator

`build_model` reads the `model` section of the configuration and returns the `nn.Module`.
`branch_1` is the state branch (an MLP on $x(t_n)$), `branch_2` is the memory branch
(MLP &rarr; LSTM &rarr; MLP on the masked history), `trunk` is an MLP on $h$, and the three
feature vectors are fused by a dot product plus the learned bias `tau` (paper Eq. 4-5).

A fraction `training.holdout_size` of the *trajectories* is held out before the quartets are
built, so no sub-sequence of a held-out trajectory is ever seen during training.

In [ ]:
set_seed()
device = resolve_device(config.training.device, verbose=True)

train_only, _ = split_dataset(train_raw, test_size=config.training.holdout_size)
train_dataset, _, state_feature_num = prepare_torch_dataset(
    train_only,
    preparer,
    state_component=config.training.state_component,
    search_len=config.training.search_len,
    search_num=config.training.search_num,
    search_random=config.training.search_random,
    offset=config.training.offset,
    t_max=config.training.t_max,
    scale_mode=config.training.scale_mode,
    device=device,
)

model = build_model(config.model, state_feature_num)
print(model)
print(f"\n{sum(p.numel() for p in model.parameters()):,} trainable parameters")

## 4. Train

`train_adam` needs an active MLflow run: it logs `train_loss`/`val_loss` every epoch, saves the
best checkpoint whenever `monitor_metric` improves, and finally logs the best model under the
artifact name `model`. `ReduceLROnPlateau` and early stopping come from the configuration.

```bash
blstm-mionet train --config configs/lorentz.yaml \
    --data data/lorentz_demo_train.npy --epochs 5 --device cpu --no-plot
# prints "MLflow run uri: runs:/<id>" and "Model uri: runs:/<id>/model"
```

In [ ]:
tracking.configure_tracking(config.tracking.uri)
with tracking.start_run(config.training.experiment_name, config.training.run_name) as run:
    RUN_ID = run.info.run_id
    tracking.log_config(config)
    history = train_adam(config.training, model, train_dataset, device)

MODEL_URI = history["model_uri"]
print(f"\nbest train_loss : {history['best_metric']['train_loss']:.5f}")
print(f"best val_loss   : {history['best_metric']['val_loss']:.5f} (epoch {history['best_epoch']})")
print(f"MLflow run uri  : runs:/{RUN_ID}")
print(f"model uri       : {MODEL_URI}")

## 5. Evaluate: one-step-ahead prediction

`evaluate_single_step` replicates every test trajectory `inference.search_num` times with
`search_random = False`, i.e. with cuts $t_n$ marching evenly along the trajectory and a fixed
spacing $h = \tfrac12 h_{\max} = 0.01$ s, and reports the relative $L_2$ error per trajectory.
This is exactly the number in Table 1 of the paper.

The model is loaded back out of MLflow, which is how `blstm-mionet infer` gets it: the run stores
the model *class* together with the weights, so nothing has to be re-instantiated by hand.

```bash
blstm-mionet infer --config configs/lorentz.yaml \
    --data data/lorentz_demo_test.npy --model runs:/<run id>/model \
    --set inference.search_num=20 --device cpu
```

In [ ]:
inference = config.inference
inference.datafile = TEST_FILE

_, test_split = split_dataset(load_dataset(TEST_FILE), test_size=1.0)
test_dataset, _, _ = prepare_torch_dataset(
    test_split,
    preparer,
    state_component=inference.state_component,
    search_len=inference.search_len,
    search_num=inference.search_num,
    search_random=inference.search_random,
    offset=inference.offset,
    t_max=inference.t_max,
    scale_mode=inference.scale_mode,
    device=device,
)

reloaded = tracking.load_model(MODEL_URI, device)
result = evaluate_single_step(inference, reloaded, test_dataset)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, idx in zip(axes, [0, 1], strict=True):
    order = np.argsort(result["t_next"][idx])
    ax.plot(result["t_next"][idx][order], result["y_true"][idx][order],
            color="red", lw=1.4, label="True")
    ax.plot(result["t_next"][idx][order], result["y_pred"][idx][order],
            color="blue", lw=1.4, ls="--", label="LSTM-MIONet")
    ax.set_xlabel("$t$ (s)")
    ax.set_ylabel("$x(t)$")
    ax.set_title(f"test trajectory {idx}: $L_2$ = {result['L2_error_mat'][idx] * 100:.2f} %")
    ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

## 6. Recursive rollout with teacher forcing

In a rollout the operator is applied autoregressively: the prediction of one step becomes the
current state of the next, and because the Lorenz system is autonomous (`autonomous = True`) the
masked *history* fed to the LSTM branch is rebuilt from the predictions as well. The
`teacher_forcing_prob` controls how often the true state is substituted instead:

* `1.0` &mdash; always the true state, i.e. the one-step-ahead error of step 5 chained together;
* `0.5` &mdash; half of the steps are corrected;
* `0.0` &mdash; a free-running simulation of a chaotic system, which is the hard case.

A rollout needs a **single** trajectory, replicated once per time step: with
`search_random = False` and `search_num` set to `N_time - 2 * search_len`, the cuts $t_n$ land on
consecutive grid indices and the rollout walks the trajectory one step at a time.

```bash
blstm-mionet infer --config configs/lorentz.yaml \
    --data data/lorentz_demo_rollout.npy --model runs:/<run id>/model \
    --recursive --teacher-forcing-prob 0.5 \
    --set inference.search_num=195 --set inference.search_random=false --device cpu
```

In [ ]:
_, rollout_split = split_dataset(rollout_raw, test_size=1.0)

rollout_cfg = copy.deepcopy(config.inference)
rollout_cfg.datafile = ROLLOUT_FILE
rollout_cfg.recursive = True
rollout_cfg.autonomous = True
rollout_cfg.search_random = False
rollout_cfg.search_num = rollout_split[1].shape[1] - 2 * rollout_cfg.search_len
rollout_cfg.plot_trajs = False
rollout_cfg.verbose = False

rollout_dataset, _, _ = prepare_torch_dataset(
    rollout_split,
    preparer,
    state_component=rollout_cfg.state_component,
    search_len=rollout_cfg.search_len,
    search_num=rollout_cfg.search_num,
    search_random=False,
    offset=rollout_cfg.offset,
    t_max=rollout_cfg.t_max,
    scale_mode=rollout_cfg.scale_mode,
    device=device,
    verbose=False,
)

rollouts = {}
for prob in (1.0, 0.5, 0.0):
    rollout_cfg.teacher_forcing_prob = prob
    print(f"--- teacher forcing probability {prob} ---")
    rollouts[prob] = evaluate_recursive(rollout_cfg, reloaded, rollout_dataset)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, prob in zip(axes, (1.0, 0.5, 0.0), strict=True):
    res = rollouts[prob]
    ax.plot(res["t_next"][0], res["y_true"][0], color="red", lw=1.3, label="True")
    ax.plot(res["t_next"][0], res["y_pred"][0], color="blue", lw=1.3, ls="--", label="Pred")
    ax.set_title(f"teacher forcing {prob}\n$L_2$ = {res['L2_error_mat'][0] * 100:.2f} %")
    ax.set_xlabel("$t$ (s)")
    ax.set_ylabel("$x(t)$")
    ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

## Reproducing the paper

Set `QUICK = False` above, or run the command line version, which is what the published numbers
come from (`configs/lorentz.yaml` holds the Section 4.1 hyper-parameters unchanged):

```bash
# 5000 trajectories of T = 20 s, Delta = 0.01 s  ->  N_train = 20000 quartets
blstm-mionet generate --config configs/lorentz.yaml

# 100 test trajectories and one trajectory for the rollout
blstm-mionet generate --config configs/lorentz.yaml --n-sample 100 \
    --output data/lorentz_N_100_h001_T20.npy
blstm-mionet generate --config configs/lorentz.yaml --n-sample 1 \
    --output data/lorentz_N_1_h001_T20.npy

# 1000 epochs of Adam with early stopping; registers models:/lorentz/latest
blstm-mionet train --config configs/lorentz.yaml --device 0

# Table 1: 200 sub-sequences on each of the 100 test trajectories
blstm-mionet infer --config configs/lorentz.yaml --model models:/lorentz/latest

# Recursive rollout, teacher forcing 1.0 / 0.5 / 0.0
blstm-mionet infer --config configs/lorentz.yaml \
    --data data/lorentz_N_1_h001_T20.npy --model models:/lorentz/latest \
    --recursive --teacher-forcing-prob 0.0 --set inference.search_num=1995

# Baseline of Table 7: the same command with --set model.architecture=DeepONet
```

Published results for the Lorenz system:

| | $x(t)$ | $y(t)$ |
| --- | --- | --- |
| mean $L_2$ (Table 1) | 1.29 % | 1.13 % |
| st. dev. $L_2$ (Table 1) | 0.93 % | 0.70 % |
| DeepONet baseline, mean $L_2$ (Table 7) | 5.23 % | &mdash; |
| reSGLD ensemble, $M = 300$: $L_2$ / PICP (Table 2) | 5.83 % / 100.00 % | 8.33 % / 97.00 % |

The uncertainty column is the subject of [`04_uncertainty.ipynb`](04_uncertainty.ipynb).

Every run above is recorded in `mlruns/`. To browse the parameters, the loss curves and the stored
models:

```bash
MLFLOW_ALLOW_FILE_STORE=true uv run mlflow ui --backend-store-uri mlruns
```

then open <http://127.0.0.1:5000>.